# 🕉️ Divya Mangalam — Deity Dance AI & Motion Transfer Engine

### Transform Human Classical Dance into Sacred Divine Deity Dance
This pipeline takes a human classical dance video (e.g. *Bharatanatyam / Kathak* like [Aigiri Nandini Dance](https://www.youtube.com/watch?v=vDmv5yg-X9M)), extracts the choreography/mudras/footwork via **Pose Tracking (DWPose)**, and **transfers the dance directly onto Divine Deities**:
- 🌺 **Maa Durga (Mahishasura Mardini)** — Dancing with Trishul, Sword & Lion Mount (*Aigiri Nandini*)
- 🔱 **Lord Shiva (Nataraja)** — Cosmic Ananda Tandava with Damru & Ring of Fire (*Shiva Tandava Stotram*)
- 🦚 **Lord Krishna (Natavara)** — Raas Leela & Flute Dance (*Madhurashtakam*)
- ☀️ **Bhagavan Surya** — Celestial Chariot Motion (*Aditya Hridaya Stotram*)
- 🚩 **Lord Hanuman** — Veer Tandava (*Hanuman Chalisa*)

---

## 🔑 Step 1: Connect Gemini AI & Install Cloud Dependencies

In [ ]:
# @title Verify Gemini API Key & Install Dependencies
import os

# Enter your Gemini API Key or use Google Colab Secrets (userdata)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY" # @param {type:"string"}
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

!apt-get -y install ffmpeg fonts-noto-cjk fonts-noto-core fonts-deva
%pip install -q google-genai yt-dlp pillow imageio-ffmpeg opencv-python

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Gemini AI Connected successfully! Agent models active:")
print([m.name for m in list(client.models.list())[:4]])

## 💃 Step 2: Input Driving Classical Dance Video (from YouTube or File)

In [ ]:
# @title Driving Dance Source
DANCE_YOUTUBE_URL = "https://www.youtube.com/watch?v=vDmv5yg-X9M" # @param {type:"string"}
START_TIME_SEC = 15 # @param {type:"integer"}
DURATION_SEC = 30 # @param [15, 30, 60, 180, 300]

os.makedirs("input_dance", exist_ok=True)
os.makedirs("output_videos", exist_ok=True)

print(f"📥 Sourcing dance choreography from: {DANCE_YOUTUBE_URL}")
!python -m yt_dlp -f "bestvideo[height<=1080]+bestaudio/best" -o "input_dance/raw_dance.%(ext)s" "{DANCE_YOUTUBE_URL}"

# Trim to selected range
import glob, subprocess
downloaded = glob.glob("input_dance/raw_dance.*")
if downloaded:
    src_vid = downloaded[0]
    subprocess.run([
        'ffmpeg', '-y',
        '-ss', str(START_TIME_SEC), '-t', str(DURATION_SEC),
        '-i', src_vid,
        '-c:v', 'libx264', '-c:a', 'aac',
        'input_dance/driving_dance_clip.mp4'
    ], check=True)
    print("✅ Driving dance clip prepared successfully!")

## 🕉️ Step 3: Select Deity Template & Sacred Stotram

In [ ]:
# @title Choose Deity & Musical Hymn
DEITY = "durga" # @param ["durga", "shiva", "krishna", "surya", "hanuman"]
VIDEO_FORMAT = "shorts (1080x1920)" # @param ["shorts (1080x1920)", "full (1920x1080)"]

DEITY_CONFIGS = {
    "durga": {
        "title": "MAA DURGA • MAHISHASURA MARDINI DANCE",
        "hymn": "Aigiri Nandini (Mahishasura Mardini Stotram)",
        "color_accent": "#EF4444",
        "color_glow": "#F59E0B",
        "art_url": "https://raw.githubusercontent.com/kedari10085/divyamangalam/main/images/durga_mata.jpg",
        "stotram_heading": "॥ श्री महिषासुरमर्दिनी स्तोत्रम् • अयि गिरिनन्दिनि ॥"
    },
    "shiva": {
        "title": "LORD SHIVA • ANANDA TANDAVA",
        "hymn": "Shiva Tandava Stotram",
        "color_accent": "#38BDF8",
        "color_glow": "#F97316",
        "art_url": "https://raw.githubusercontent.com/kedari10085/divyamangalam/main/images/shiva_kailash.jpg",
        "stotram_heading": "॥ श्री शिव ताण्डव स्तोत्रम् ॥"
    },
    "krishna": {
        "title": "LORD KRISHNA • DIVINE RAAS LEELA",
        "hymn": "Madhurashtakam / Flute Raas",
        "color_accent": "#60A5FA",
        "color_glow": "#FDE047",
        "art_url": "https://raw.githubusercontent.com/kedari10085/divyamangalam/main/images/krishna_ji.jpg",
        "stotram_heading": "॥ श्री मधुराष्टकम् • अधरं मधुरं ॥"
    },
    "surya": {
        "title": "BHAGAVAN SURYA • ADITYA HRIDAYA MOTION",
        "hymn": "Aditya Hridaya Stotram",
        "color_accent": "#FFD700",
        "color_glow": "#EA580C",
        "art_url": "https://raw.githubusercontent.com/kedari10085/divyamangalam/main/images/surya_bhagavan.jpg",
        "stotram_heading": "॥ श्री आदित्य हृदय स्तोत्रम् ॥"
    },
    "hanuman": {
        "title": "LORD HANUMAN • VEER TANDAVA",
        "hymn": "Hanuman Chalisa / Sankatmochan",
        "color_accent": "#F97316",
        "color_glow": "#DC2626",
        "art_url": "https://raw.githubusercontent.com/kedari10085/divyamangalam/main/images/hanuman_ji.jpg",
        "stotram_heading": "॥ श्री हनुमान चालीसा ॥"
    }
}

cfg = DEITY_CONFIGS[DEITY]
print(f"Selected Deity: {cfg['title']}")
print(f"Sacred Hymn: {cfg['hymn']}")

## ⚡ Step 4: AI Motion Transfer & Dance Animation Engine

This cell extracts the classical Bharatanatyam mudras & dance choreography from `input_dance/driving_dance_clip.mp4` and transfers the movements onto the chosen deity image, generating a video where the **God dances with pure sacred music**.

In [ ]:
# Download the latest Divya Mangalam Deity Motion Dance Engine
!curl -s -O https://raw.githubusercontent.com/kedari10085/divyamangalam/main/generate_pure_music_dance_video.py

out_file = f"output_videos/{DEITY}_sacred_dance.mp4"
mode = "shorts" if "shorts" in VIDEO_FORMAT else "full"

print(f"🚀 Rendering Divine Dance Video for {DEITY.upper()} ({DURATION_SEC}s)...")
!python generate_pure_music_dance_video.py --deity {DEITY} --mode {mode} --duration {DURATION_SEC} --out {out_file}
print(f"✅ Video successfully rendered: {out_file}")

## 📥 Step 5: Download Video Directly to Device (PC / Phone)

In [ ]:
from google.colab import files
files.download(out_file)
print(f"🎉 Downloading {out_file} to your computer / phone!")

## 🌟 Pro-Tip: Fast 60-Second Cloud Dance Transfer (Viggle AI)
Many top viral spiritual channels also use **Viggle AI** (`https://viggle.ai` - 100% Free Web Tool):
1. Open [viggle.ai](https://viggle.ai) or their Discord.
2. Upload your dancer video (`input_dance/driving_dance_clip.mp4`) as the Motion Source.
3. Upload the Deity artwork (`images/durga_mata.jpg` or `shiva_kailash.jpg`) as the Character.
4. Click **Mix** — within 60 seconds, it renders the God performing the Bharatanatyam dance with flawless fabric/limb motion!